# How to Access Pipeline Results - Deep Dive

Each step in the pipeline writes its results in the `Result` object of the pipeline instance (Vanillix, Varix, etc.).  
In this tutorial, we explore how to access and interpret the results.  

The attributes of the `Result` object are mostly instances of a `TrainingDynamics` class.  
This class provides a standardized interface for accessing results from different splits and epochs.  

**IMPORTANT**  
> Epoch-specific `TrainingDynamics`—such as losses or intermediate latent spaces—are not stored every epoch by default.  
> You need to set the `checkpoint_interval` parameter in the config according to your needs.

## What You Will Learn

We go in depth into:

- The `TrainingDynamics` API  
  - latent spaces  
  - losses  
  - reconstructions  
  - sample_ids  

- Nested `TrainingDynamics` like `sub_losses`  

- Non-`TrainingDynamics` result attributes, such as:  
  - datasets  
  - new_datasets  
  - model  
  - adata_latent  
  - final_reconstruction  
  - embedding_evaluation  

- Special methods to obtain pandas DataFrames  
  - get latent space as a DataFrame with `sample_ids`  
  - get reconstruction as a DataFrame with `sample_ids`  

## 1) Filling the Result Object

Before we can investigate the result object, we first need to create results.  
Therefore, we run two pipelines: `XModalix` and `Varix`.

### 1.1 The Datasets

For the `Varix` example, we use a mock single-cell dataset as a `MuData` object inside our custom `DataPackage` class.  
For our `XModalix` example, we use the same dataset as in the `XModalix.ipynb` tutorial.  

As a showcase for data modality translation with `XModalix`, we use cancer gene expression from TCGA in combination with handwritten digits from the [MNIST dataset](https://keras.io/api/datasets/mnist/).  
Our goal is to translate the gene expression signature of five selected cancer subtypes to images of digits, where each cancer subtype class is assigned a digit between 0-4.  

In practice, these images could be histopathological images or any other data modality.  
Before showing data preparation and `XModalix` training, here is some background on the basic idea of a cross-modal VAE as proposed by [Yang & Uhler](https://arxiv.org/abs/1902.03515).

### ❗❗ Requirements: Getting Tutorial Data ❗❗

You can use the following bash commands to download the data and set up the correct folder structure.  
**Assumption:** you are in the root of `autoencodix_package`.

```bash
mkdir -p data
cd data
wget "https://cloud.scadsai.uni-leipzig.de/index.php/s/bq64MaQyZGZfN64/download/XModalix-Tut-data.zip"
unzip XModalix-Tut-data.zip
```



In [1]:
import os

p = os.getcwd()
d = "autoencodix_package"
if d not in p:
    raise FileNotFoundError(f"'{d}' not found in path: {p}")
os.chdir(os.sep.join(p.split(os.sep)[: p.split(os.sep).index(d) + 1]))
print(f"Changed to: {os.getcwd()}")


Changed to: /Users/maximilianjoas/development/autoencodix_package


In [3]:
# %%capture
from autoencodix.utils.example_data import EXAMPLE_MULTI_SC
from autoencodix.configs.varix_config import VarixConfig
from autoencodix.configs.default_config import DataCase, DataInfo, DataConfig
from autoencodix.configs.xmodalix_config import XModalixConfig
import autoencodix as acx

varix_config = VarixConfig(
    learning_rate=0.001,
    epochs=33,
    checkpoint_interval=1,
    default_vae_loss="kl",  # kl or mmd possible
    data_case=DataCase.MULTI_SINGLE_CELL,
)
varix = acx.Varix(data=EXAMPLE_MULTI_SC, config=varix_config)
result = varix.run()

# XModalix
clin_file = os.path.join("data/XModalix-Tut-data/combined_clin_formatted.parquet")
rna_file = os.path.join("data/XModalix-Tut-data/combined_rnaseq_formatted.parquet")
img_root = os.path.join("data/XModalix-Tut-data/images/tcga_fake")

xmodalix_config = XModalixConfig(
    checkpoint_interval=1,
    class_param="CANCER_TYPE_ACRONYM",
    epochs=30,
    device="cpu",
    data_case=DataCase.IMG_TO_BULK,
    data_config=DataConfig(
        data_info={
            "img": DataInfo(
                file_path=img_root,
                data_type="IMG",
                scaling="MINMAX",
                translate_direction="to",
            ),
            "rna": DataInfo(
                file_path=rna_file,
                data_type="NUMERIC",
                scaling="MINMAX",
                translate_direction="from",
            ),
            "anno": DataInfo(file_path=clin_file, data_type="ANNOTATION", sep="\t"),
        },
    ),
)

xmodalix = acx.XModalix(config=xmodalix_config)
xmodalix_result = xmodalix.run()

in handle_direct_user_data with data: <class 'autoencodix.data.datapackage.DataPackage'>
mudata: View of MuData object with n_obs × n_vars = 1000 × 700
  2 modalities
    rna:	1000 x 500
      obs:	'cell_type', 'batch', 'donor', 'cell_cycle', 'n_genes'
    protein:	1000 x 200
      obs:	'cell_type', 'batch', 'donor', 'cell_cycle', 'n_genes'
Processing 1 MuData objects: ['multi_sc']
Processing train modality: multi_sc
Processing valid split
Processing valid modality: multi_sc
Processing test split
Processing test modality: multi_sc
Epoch 1 - Train Loss: 781.3026
Sub-losses: recon_loss: 781.3026, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 1 - Valid Loss: 642.8199
Sub-losses: recon_loss: 642.8199, var_loss: 0.0001, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 2 - Train Loss: 678.0888
Sub-losses: recon_loss: 678.0888, var_loss: 0.0001, anneal_factor: 0.0001, effective_beta_factor: 0.0000
Epoch 2 - Valid Loss: 574.2986
Sub-losses: recon_loss: 

KeyboardInterrupt: 

## 2) TrainingDynamics Interface Deep Dive
Before accessing the actual results, we provide a theory section on our interface:  
<br><br>
The `TrainingDynamics` object has the following form:  
`<epoch><split><data>`  

So, if you want to access the train loss for the 5th epoch, you would use:  
```python
result.loss.get(epoch=5, split="train")
````

##### The `.get()` Method Explained
Let's say, we're interessted in thre reconstructions of our autoencoder.  
The `reconstructions.get()` method provides flexible access to reconstruction data stored during training. It can retrieve data for specific epochs, specific splits, or any combination of these parameters.

##### Parameters

* **`epoch`** (Optional[int]):

  * Positive integer (e.g., `2`): Get reconstructions from that specific epoch
  * Negative integer (e.g., `-1`): Get the latest epoch (-1), second-to-last (-2), etc.
  * `None`: Return data for all epochs

* **`split`** (Optional[str]):

  * Valid values: `"train"`, `"valid"`, `"test"`
  * `None`: Return data for all splits

##### Return Value Behavior

The method returns different types depending on the parameters:

1. **Both `epoch` and `split` specified**:

   * Returns a NumPy array for that specific epoch and split
   * Example: `get(epoch=2, split="train")` → `array([...])`

2. **Only `epoch` specified**:

   * Returns a dictionary of all splits for that epoch
   * Example: `get(epoch=2)` → `{"train": array([...]), "valid": array([...]), ...}`

3. **Only `split` specified**:

   * Returns a NumPy array containing data for that split across all epochs
   * Example: `get(split="train")` → `array([[...], [...], ...])` (first dimension represents epochs)

4. **Neither specified**:

   * Returns the complete nested dictionary structure
   * Example: `get()` → `{0: {"train": array([...])}, 1: {...}, ...}`

##### Special Handling

* If an invalid split is provided, a `KeyError` is raised
* Negative epoch indices work like Python list indexing (-1 is the last epoch)
* If an epoch doesn't exist, an empty array or dictionary is returned

##### Code Example

```python
# Access train reconstructions for the 5th epoch
train_epoch_5 = result.reconstructions.get(epoch=5, split="train")

# Access all splits for the latest epoch
latest_epoch_all_splits = result.reconstructions.get(epoch=-1)

# Access data for all epochs for the "valid" split
all_epochs_valid = result.reconstructions.get(split="valid")

# Access the full nested dictionary
full_data = result.reconstructions.get()
```


## 3) Working with Actual Results
### 3.1) Varix

Exemplary, we show how to get the `latentspaces` of the `result` attribute and to access different loss types.

In [ ]:
all_ls = result.latentspaces.get()
print(f"Keys of all latentspaces: {all_ls.keys()}")


Keys of all latentspaces: dict_keys([-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32])


We see that we have latent spaces for each epoch because we set `checkpoint_interval=1` in our configs in [Step 1](#1-filling-the-result-object).  

For each epoch, we have the latent space for the `train` and `valid` splits. The `-1` epoch is a special key for the `test` split. For the other splits, negative indexing works as in Python lists: `-1` gives the last epoch, `-2` the second-to-last, and so on.  

**Special Case**  
> You cannot get the last epoch for all splits at once. You can either get the last epoch for `train` and `valid`, or only for `test`. See code below.


In [ ]:
print(f"Splits in 2. epoch: {result.latentspaces.get(epoch=2).keys()}")

# this will only give you the data for train and valid, since 'test' is a special case
print(f"Splits in epoch =-1: {result.latentspaces.get(epoch=-1).keys()}")
# Get test by adding the 'split' argument.
test_ls = result.latentspaces.get(split="test")
print(test_ls[0][0])
# get a specific epoch and specific split:

print("\n")
print("-"*80)

print(f"latentspace of one sample in train split at epoch 4: {result.latentspaces.get(split='train', epoch=4)[0]}")

SyntaxError: f-string: unmatched '(' (3696328712.py, line 13)

### 3.2 XModalix

Accessing the results works slightly differently for `XModalix`, because we have:

1. **Multiple latent spaces** (one for each data modality).  

2. **Translations and reference translations**, not just reconstructions.  
   - These are reconstructions within one data modality on the same split as the translation. For example, if we translate from `rna` to `img`, the reference translation would be the reconstruction from `img` to `img`.  
   - You can access these `translations` via the `reconstruction` attribute of the `result` object. First, apply the usual `TrainingDynamics` API via `get()`, then you get a dictionary for each data modality with translations and reference translations.

3. **Multiple losses**.  
   - These are accessed via the `sub_losses` attribute of the `result` object, which is a dictionary of `TrainingDynamics`. First, select the loss type you're interested in, then work with the usual `TrainingDynamics` API.  

   **Note on naming sub-losses:**  
   - Global losses are simply named after the loss type, e.g., `class_loss`.  
   - Losses per data modality are named with the following convention:  
     ```
     <global_data_modality_key>.<specific_data_modality_name>.<loss_name>
     ```  
     For example: `multi_bulk.rna.class_loss` or `multi_sc.celltype.reconstruction_loss`. See print below.


#### 3.2.1 Access Modality Latent Spaces

As described in our `XModalix Deep Dive` [1], we fit one latent space per data modality.  
You can access this by first selecting the epoch and split you're interested in (standard `TrainingDynamics` API).  
The result will be a `Dict` with the name of each data modality as the key.


In [ ]:
print(f" Keys of data modalities for latent space dynamic: {xmodalix_result.latentspaces.get(epoch=-1, split='test').keys()}")


Now you can access the latent space of the `image` modality with the key `img.img`

In [ ]:
xmodalix_result.latentspaces.get(epoch=-1, split="test").get("img.img")

#### 3.2.2 Access Translation

In [ ]:
print("Get reconstruction keys")
# Frist define split and epoch you're interested in
# usually test split (there are no epochs, so by default this is always epoch=-1)
recons = xmodalix_result.reconstructions.get(split="test", epoch=-1)
print(recons.keys())
print("Getting Translation")
trans = recons.get("translation")
print(f"shape of translation: {trans.shape}")

#### 3.2.3 Access Sub-Losses

In [ ]:

sub_losses = xmodalix_result.sub_losses
print("Sub Losses:")
print(f"keys: {sub_losses.keys()}")
print("\n")
recon_dyn = sub_losses.get(key="paired_loss")
print("Value of paired loss in epoch 4 for train split")
print(f"{recon_dyn.get(split='train', epoch=4):.2f}")


## 4) Non-TrainingDynamics Result Attributes

There are other (intermediate) results that are not created during training but might still be interesting.  
These results do not follow a uniform interface like `TrainingDynamics`, but are often more straightforward. We go over each attribute quickly.

#### 4.1 Datasets

The `datasets` attribute stores the preprocessed data in a `DatasetContainer`.  
This is basically a dict with `train`, `valid`, and `test` as keys, and each value is a child class of a PyTorch dataset.  
Whenever you need to re-access your preprocessed data, you can do so using the `datasets` attribute, as shown below:


In [ ]:
print(result.datasets)
print(result.datasets.train)
print(type(result.datasets.train.data))
result.datasets.train.metadata.head()

#### 4.2 New Datasets

Whenever you run the `predict` step of the pipeline and pass new, unseen data to it, we preprocess this data (if necessary).  
To avoid overwriting the original `datasets`, we store this in `new_datasets`.  

If you run `predict` again with other data, `new_datasets` will be overridden.  
Otherwise, `new_datasets` works the same way as `datasets`.


First we create new data and then we run predict.

In [ ]:
import copy

from autoencodix.utils.example_data import EXAMPLE_MULTI_SC
new_data = copy.copy(EXAMPLE_MULTI_SC)
new_multi_sc = new_data.multi_sc["multi_sc"]
for modname, mod in new_multi_sc.mod.items():
    new_names = mod.obs_names.str.replace('cell', 'new_cell')
    mod.index = new_names

    mod.obs_names = new_names
    print(mod.index)
    new_multi_sc.mod[modname] = mod

new_data.multi_sc["multi_sc"] = new_multi_sc
new_data.multi_sc["multi_sc"].update()

Run the predict step:

In [ ]:
%%capture
varix.predict(data=new_data)

Examine `datasets` and `new_datasets`:  
We see that `datasets` still is kept and only new_datasets is updated.

In [ ]:
print(f"Sample of original dataset: {result.datasets.train.sample_ids[0]}")

print(f"Sample of new dataset: {result.new_datasets.test.sample_ids[0]}")

#### 4.3 Model Attribute
This is straightforward the trained model as PytTorch Module.

In [ ]:
result.model

#### 4.4 Adata Latent

We save the latent space of the `test` split from the final trained model as an `AnnData` object for the single-cell community.  

This is also useful for non-single-cell cases, because you can still obtain the sample IDs via `.obs`.


In [ ]:
print(result.adata_latent)
print(result.adata_latent.obs)

#### 4.5 Final Reconstruction
This attribute gives you the exact data structure as you used for input i.e. `MuData` in our case, with the reconstructed values.

In [ ]:
result.final_reconstruction

#### 4.6 Evaluation Embeddings

Before we can access this attribute, we first need to run the `evaluate` step. This will use the latent space and train a downstream machine learning task. In our case, we want to classify the cancer type.  

The results of this evaluate step will be stored in `embedding_evaluation`.


In [ ]:
%%capture
xmodalix.evaluate(params=["CANCER_TYPE"])

In [ ]:
xmodalix_result.embedding_evaluation

In [ ]:

varix.result.datasets.test.metadata.head()
varix.evaluate(params=["rna:batch"])

##  5 Special Methods to Obtain DataFrames
As you've  seen in the [Training Dynamics Section](#2-trainingdynamics-interface-deep-dive), we only get plain values of reconstructions and latetnspaces. Often it is more useful to have sample ids, too. We could obtain the sample ids in the same order via the `sample_ids` TrainingDynamic. To make this more accessible, we added the methods
`get_latent_df` and `get_reconstructions_df`. Here you pass `epoch` and `split` as seen before and you get a pandas DataFrame for the specific split and epoch for the latent space or the reconstruction.

In [ ]:
result.get_latent_df(epoch=-1, split="test").head()

In [ ]:
result.get_reconstructions_df(epoch=-1, split="test").head()